In [ ]:
REPO = 'zl910938678/inkling-2-mini-kaggle'
REF = 'main'
PRIVATE_REPO = True

import base64
import os
from pathlib import Path
import re
import runpy
import subprocess
from tempfile import TemporaryDirectory

if not re.fullmatch(r'[A-Za-z0-9-]+/[A-Za-z0-9_.-]+', REPO) or REPO.startswith('OWNER/'):
    raise ValueError('Set REPO to the GitHub repository path.')
with TemporaryDirectory(prefix='inkling-src-') as checkout:
    env = dict(os.environ, GIT_TERMINAL_PROMPT='0', GIT_CONFIG_COUNT='0')
    if PRIVATE_REPO:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_READ_TOKEN')
        auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env.update(GIT_CONFIG_COUNT='1', GIT_CONFIG_KEY_0=f'http.https://github.com/{REPO}.git.extraHeader', GIT_CONFIG_VALUE_0=f'Authorization: Basic {auth}')
    subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--single-branch', '--branch', REF, f'https://github.com/{REPO}.git', checkout], env=env, check=True)
    runpy.run_path(str(Path(checkout) / 'serve.py'), run_name='__main__')
